In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import os
from typing import Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import  HumanMessage
from pydantic import BaseModel, Field

# 读取env文件,将env内容加载到系统环境变量
load_dotenv()

# 读取apikey和模型
dashscope_api_key = os.getenv('DASHSCOPE_API_KEY')
dashscope_base_url = os.getenv('DASHSCOPE_BASE_URL')

# 初始化模型
model = init_chat_model(
    model='qwen3.8-max',
    base_url=dashscope_base_url,
    api_key=dashscope_api_key,
    model_provider='openai',
    temperature=1,
    top_p=1,
    # 额外参数
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 创建Mcp客户端
mcp_client = MultiServerMCPClient(
    {
        "time-mcp": {
            # 额外需要指定通信协议
            "transport": "stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        },
        "12306-mcp": {
            # 使用远程调用
          "transport": "streamable_http",
          "url": "https://mcp.api-inference.modelscope.net/d09335d86eac49/mcp"
        }
    }
)
mcp_tools = await mcp_client.get_tools()

# 创建智能体
my_agent = create_agent(tools=mcp_tools,model=model)

# 调用大模型
response = await my_agent.ainvoke({
    "messages":[
        HumanMessage(content='帮我查询一下明天中午12点到晚上六点间武汉去重庆的高铁票')
    ]
})




In [3]:
print(response["messages"][-1].content)

为您查询到明天（2026-08-29）中午12点至晚上6点间，武汉到重庆的高铁票信息如下：

| 车次 | 出发站 -> 到达站 | 出发时间 -> 到达时间 | 历时 | 余票及价格 |
| :--- | :--- | :--- | :--- | :--- |
| G3455 | 汉口 -> 重庆北 | 12:30 -> 17:26 | 04:56 | 商务座(6张/¥1565)，一等座(无票/¥772)，二等座(无票/¥482)，无座(有票/¥482) |
| G3391 | 汉口 -> 重庆北 | 12:32 -> 20:11 | 07:39 | 商务座(9张/¥1869)，一等座(12张/¥854)，二等座(有票/¥533)，无座(有票/¥533) |
| G3481 | 汉口 -> 重庆北 | 13:53 -> 18:50 | 04:57 | 商务座(5张/¥1596)，一等座(无票/¥788)，二等座(12张/¥502)，无座(有票/¥502) |
| G3457 | 汉口 -> 重庆北 | 17:00 -> 21:57 | 04:57 | 商务座(10张/¥1513)，一等座(2张/¥750)，二等座(有票/¥468)，无座(有票/¥468) |

**温馨提示：**
*   以上车次均从**汉口站**出发，抵达**重庆北站**。
*   G3455和G3481的二等座票源较为紧张或已售罄，如需乘坐该时段车次建议尽早预订其他席别或选择G3391、G3457。
*   实际余票情况可能随时变动，请以12306官方信息为准。
